In [1]:
import pandas as pd 
import geopandas as gpd
import pyarrow
import pyogrio

print(pyarrow.__version__)
print(pyogrio.__version__)


15.0.2
0.12.1


In [14]:
gdf_aoi = gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1)
# gdf_aoi = gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1).total_bounds
# gdf_aoi.to_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/outputs/2026-01-philippines-angeles/02-process-output/spatial/angeles_globfire_aoi_buffered.gpkg', driver='GPKG')

/var/folders/_j/40dgp1fx17bb2bwpyzg2rbc80000gn/T/ipykernel_43107/1010530353.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_aoi = gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1)


In [2]:
gdf = gpd.read_file('https://storage.googleapis.com/city-scan-global-public/globfire_fgb/MODIS_BA_GLOBAL_1_11_2000.fgb')
gdf.shape

(68467, 5)

In [ ]:
gdf.to_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/outputs/2026-01-philippines-angeles/02-process-output/spatial/angeles_globfire_test.gpkg', driver='GPKG')

In [18]:
import pyarrow

gdf_bbox = tuple(gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1).total_bounds)

gdf2 = gpd.read_file('https://storage.googleapis.com/city-scan-global-public/globfire_fgb/MODIS_BA_GLOBAL_1_11_2002.fgb', 
                     bbox=gdf_bbox,
                     engine='pyogrio',
                     use_arrow = True,
                     where="Type = 'FinalArea'",
                     )
gdf2.shape

/var/folders/_j/40dgp1fx17bb2bwpyzg2rbc80000gn/T/ipykernel_43107/4258059719.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_bbox = tuple(gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1).total_bounds)


(18, 5)

In [19]:
map = gdf2.explore()
gdf_aoi.boundary.explore(m=map)

In [ ]:
import os
os.environ["GDAL_CACHEMAX"] = "512"  # MB


gdf_bbox = tuple(gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1).total_bounds)

gdf2 = gpd.read_file('/vsicurl/https://storage.googleapis.com/city-scan-global-public/globfire/MODIS_BA_GLOBAL_1_9_2012.shp', 
                     bbox=gdf_bbox,
                     engine='pyogrio',
                     use_arrow = True,
                     where="Type = 'FinalArea'",
                     on_invalid="ignore"
                     )
gdf2.shape

In [ ]:
import os
os.environ["GDAL_CACHEMAX"] = "512"  # MB


gdf_bbox = tuple(gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1).total_bounds)

gdf3 = gpd.read_file('/vsicurl/https://storage.googleapis.com/city-scan-global-public/globfire/MODIS_BA_GLOBAL_1_9_2016.shp', 
                     bbox=gdf_bbox,
                     engine='pyogrio',
                     use_arrow = True,
                     where="Type = 'FinalArea'",
                     on_invalid="ignore", 
                     columns=[]
                     )
gdf3.shape

In [ ]:
gdf3

In [ ]:
years = range(2018, 2019)
for year in years:
    print(year)

In [ ]:
import time
from tqdm import tqdm
import pandas as pd
import geopandas as gpd
import os
os.environ["GDAL_CACHEMAX"] = "512"  # MB



gdf_bbox = tuple(gpd.read_file('/Users/danielcp/local_drive/01_CRP/city-scan-automation/inputs/AOI/Angeles-Pampanga.shp').buffer(1).total_bounds)
records = []

years = range(2018, 2019)
months = range(1, 13)

start_total = time.perf_counter()
file_counter = 0

for year in years:
    for month in tqdm(months, desc=f"Year {year}"):

        t0 = time.perf_counter()

        path = f"/vsicurl/https://storage.googleapis.com/city-scan-global-public/globfire/MODIS_BA_GLOBAL_1_{month}_{year}.shp"

        gdf = gpd.read_file(
            path,
            engine="pyogrio",
            use_arrow=True,
            bbox=gdf_bbox,
            where="Type = 'FinalArea'",
            columns=[]
        )

        t_read = time.perf_counter()

        if len(gdf) == 0:
            print(f"[{year}-{month:02}] empty | read {t_read - t0:.2f}s")
            continue

        cent = gdf.geometry.centroid

        records.extend(
            zip(
                [year] * len(cent),
                [month] * len(cent),
                cent.x,
                cent.y
            )
        )

        t_end = time.perf_counter()

        file_counter += 1
        elapsed_total = t_end - start_total
        avg_per_file = elapsed_total / file_counter

        print(
            f"[{year}-{month:02}] "
            f"features={len(gdf):,} | "
            f"read={t_read - t0:.2f}s | "
            f"centroid={t_end - t_read:.2f}s | "
            f"total={t_end - t0:.2f}s | "
            f"avg/file={avg_per_file:.2f}s"
        )

df = pd.DataFrame(records, columns=["year", "month", "x", "y"])

print(f"\nTOTAL TIME: {time.perf_counter() - start_total:.2f}s")
print(f"TOTAL RECORDS: {len(df):,}")
